In [5]:
import numpy as np
import matplotlib.pyplot as plt
from Utilities import extractor
import uproot
import awkward as ak    

x_MH75=extractor("Dati/Tprime_tAq_1800_MH75_LH_2017.root", "Events")


file=uproot.open("Dati/Tprime_tAq_1800_MH75_LH_2017.root")
tree=file["Events"]
booleans= tree.arrays(["FatJet_isMatchedWithA"], library="ak")
booleanas=tree.arrays(["FatJet_isMatchedWith2BHadrons"], library="ak")
Fatjet_isMatchedWithA= booleans["FatJet_isMatchedWithA"]
Fatjet_isMatchedWith2BHadrons= booleanas["FatJet_isMatchedWith2BHadrons"]

#Filtriamo i dati

mask = (ak.flatten(Fatjet_isMatchedWithA) == 1) & (ak.flatten(Fatjet_isMatchedWith2BHadrons) == 1)
x_filtered = x_MH75[mask]


In [ ]:
from scipy.special import voigt_profile
from iminuit import Minuit
from iminuit.cost import LeastSquares

x_plot=list(x_filtered)
x_plot.sort()

x_easy=[x for x in x_plot if 25 <= x <= 125]

def voigt(x, norm, mu, sigma, gamma):
    return voigt_profile(x-mu, sigma, gamma) * norm 

bin_counts, bin_edges = np.histogram(x_easy, bins=50)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_width = bin_edges[1] - bin_edges[0]
bin_densities = bin_counts / (len(x_easy) * bin_width)  # Densità normalizzata
yerr=np.sqrt(bin_counts) / (len(x_easy) * bin_width) # Errore standard per i dati binned

ls_voigt=LeastSquares(bin_centers, bin_densities, yerr, model=voigt)

m_voigt=Minuit(ls_voigt,  norm=1, mu=75, sigma=5, gamma=1)
m_voigt.limits["mu"]= (50, 100)
m_voigt.limits["sigma"]= (0.1, 20)
m_voigt.limits["gamma"]= (0.01, 10)
m_voigt.migrad()



┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 1287 (χ²/ndof = 28.0)      │              Nfcn = 113              │
│ EDM = 2.13e-06 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬───────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name  │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼───────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ norm  │   1.007   │   0.004   │            │            │         │         │       │
│ 1 │ mu    │   76.43   │   0.04    │            │            │   50    │   100   │       │
│ 2 │ sigma │   5.96    │   0.06    │            │            │   0.1   │   20    │       │
│ 3 │ gamma │   2.38    │   0.05    │            │            │  0.01   │   10    │       │
└───┴───────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌───────┬─────────────────────────────────────────┐
│       │      norm        mu     sigma     gamma │
├───────┼─────────────────────────────────────────┤
│  norm │  2.01e-05  0.001e-3 -0.032e-3  0.033e-3 │
│    mu │  0.001e-3   0.00144   -0.0004    0.0000 │
│ sigma │ -0.032e-3   -0.0004   0.00409   -0.0024 │
│ gamma │  0.033e-3    0.0000   -0.0024   0.00248 │
└───────┴─────────────────────────────────────────┘

In [7]:
fit_MH75_values={}
fit_MH75_errors={}

fit_values={'MH75': fit_MH75_values,}
fit_errors={'MH75_errors': fit_MH75_errors}

for param in m_voigt.parameters:
    fit_MH75_values[param] = m_voigt.values[param]

for error in m_voigt.parameters:    #Qui non ho capito come fa a capire che deve estarre gli errori 
    fit_MH75_errors[error] = m_voigt.errors[error]

print(fit_MH75_values)
print(fit_MH75_errors)

import json
#QUi sono andato di metodo oragutang, ho deciso di voler fare 2 file separati peer errori e valori 
#Ho tenuto lo stesso quello con tutti i valori, casomai cambiassi idea

with open("fit_results.json", "r") as f:
    results=json.load(f)

with open("fit_values.json", "r") as g:
    values=json.load(g) 

with open("fit_errors.json", "r") as h:
    errors=json.load(h)


results["MH75"]=fit_MH75_values
results["MH75_errors"]=fit_MH75_errors

with open("fit_results.json", "w") as f:
    json.dump(results, f, indent=1)

values["MH75"]=fit_MH75_values
with open("fit_values.json", "w") as g:
    json.dump(values, g, indent=1)  

errors["MH75_errors"]=fit_MH75_errors
with open("fit_errors.json", "w") as h:
    json.dump(errors, h, indent=1)  


{'norm': 1.0065945568241894, 'mu': 76.42872323591135, 'sigma': 5.9582238010211395, 'gamma': 2.3810383455849182}
{'norm': 0.004483489240876725, 'mu': 0.03796783903737122, 'sigma': 0.06398521484850894, 'gamma': 0.04982423655598267}
